# HB_WEST Price Model — 5-Strategy Economic Backtest

Compares the incremental value of the price model (Layer 2) on top of the PRC model (Layer 1).

**5 strategies:**
1. Always-on — no model, eat every spike
2. PRC-only — curtail on PRC regime (current system)
3. Price-only — curtail on predicted price threshold
4. Combined — curtail when EITHER signal triggers
5. Oracle — perfect foresight (theoretical ceiling)

**Key comparison:** Strategy B vs D = incremental dollar value of adding the price model.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib
import json
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
MODELS_DIR = PROJECT_ROOT / 'models'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

## Load data and models

In [2]:
# Load model_ready data
df = pd.read_parquet(PROCESSED_DIR / 'model_ready.parquet')
print(f'Loaded: {df.shape}')

# Load price model + metadata
price_model = joblib.load(MODELS_DIR / 'lgbm_price_1h_v1.pkl')
with open(MODELS_DIR / 'lgbm_price_1h_v1_metadata.json') as f:
    price_meta = json.load(f)

price_offset = price_meta['price_offset']
price_features = price_meta['features']
print(f'Price model: {len(price_features)} features, offset={price_offset}')

# Load PRC models
lr_1h = joblib.load(MODELS_DIR / 'lr_1h_prc_v2.pkl')
lgbm_1h_residual = joblib.load(MODELS_DIR / 'lgbm_1h_residual_v2.pkl')

with open(MODELS_DIR / 'lr_1h_prc_v2_metadata.json') as f:
    prc_meta = json.load(f)
prc_features = prc_meta['features']
print(f'PRC model: {len(prc_features)} features')

Loaded: (43263, 86)


Price model: 78 features, offset=38.8475
PRC model: 71 features


## Define test set and generate predictions

Test set: Jul 2024 – Dec 2025 (matches price model test split).
Both models score the same rows so dollar comparisons are apples-to-apples.

In [3]:
# Test set: Jul 2024 onward (price model test split)
test = df[df['timestamp'] >= '2024-07-01'].copy()

# -- Price model predictions --
X_price = test[price_features].dropna()
pred_log = price_model.predict(X_price)
pred_price = np.expm1(pred_log) - price_offset

# -- PRC model predictions (LR + LGBM residual ensemble) --
# Impute ORDC NaNs for LR (LGBM handles natively)
test_lr = test.copy()
for col in ['RTORPA_log1p', 'RTOFFPA_log1p', 'RTORDPA_log1p']:
    if col in test_lr.columns:
        test_lr[col] = test_lr[col].fillna(0)

X_prc = test_lr[prc_features].dropna()
pred_prc = lr_1h.predict(X_prc) + lgbm_1h_residual.predict(X_prc)

# Align indices — only keep rows where both models have predictions
common_idx = X_price.index.intersection(X_prc.index)
print(f'Test rows: {len(test):,}, both models scored: {len(common_idx):,}')

bt = test.loc[common_idx, ['timestamp', 'RT_price', 'PRC']].copy()
bt['pred_price'] = pred_price[X_price.index.isin(common_idx)]
bt['pred_prc'] = pred_prc[X_prc.index.isin(common_idx)]

print(f'\nActual price stats:')
print(bt['RT_price'].describe())
print(f'\nSpikes >$100: {(bt["RT_price"] > 100).sum()}')
print(f'Spikes >$200: {(bt["RT_price"] > 200).sum()}')

Test rows: 12,792, both models scored: 12,144

Actual price stats:
count    12144.000000
mean        31.302737
std         46.077219
min        -30.625000
25%         16.271250
50%         25.175000
75%         37.980000
max       1963.407500
Name: RT_price, dtype: float64

Spikes >$100: 329
Spikes >$200: 62


## Backtest: Mining use case

- 200 MW load, binary on/off
- Revenue baseline: $40/MWh (mining income)
- When on: net cost = (RT_price - $40) × 200 MW per hour
- When curtailed: net cost = $0 (no power, no mining revenue, no electricity cost)

In [4]:
CAPACITY_MW = 200
MINING_REV = 40  # $/MWh mining income baseline

# PRC thresholds (from existing system)
PRC_SCARCITY = 3000
PRC_TIGHT = 5000

# Hourly net cost when ON: (electricity cost - mining revenue) * capacity
# Positive = losing money, Negative = making money
bt['hourly_net_cost'] = (bt['RT_price'] - MINING_REV) * CAPACITY_MW

# -- Strategy A: Always-on (no model) --
bt['cost_A'] = bt['hourly_net_cost']

# -- Strategy E: Oracle (perfect foresight) --
# Curtail whenever RT_price > mining revenue (would lose money staying on)
bt['cost_E'] = bt['hourly_net_cost'].clip(upper=0)  # only keep hours where we make money

print('Sanity check:')
print(f'  Hours where mining loses money (RT > ${MINING_REV}): {(bt["RT_price"] > MINING_REV).sum():,}')
print(f'  Hours where mining makes money (RT <= ${MINING_REV}): {(bt["RT_price"] <= MINING_REV).sum():,}')

Sanity check:
  Hours where mining loses money (RT > $40): 2,716
  Hours where mining makes money (RT <= $40): 9,428


In [5]:
def run_mining_backtest(bt, price_thresholds=[40, 60, 80, 100]):
    """Run all 5 strategies across multiple price curtailment thresholds."""
    
    results = []
    
    # Strategy A: Always-on
    total_A = bt['cost_A'].sum()
    results.append({
        'strategy': 'A: Always-on',
        'price_threshold': '-',
        'total_cost': total_A,
        'hours_curtailed': 0,
        'savings_vs_A': 0,
    })
    
    # Strategy B: PRC-only (current system)
    curtail_B = (bt['pred_prc'] < PRC_TIGHT)  # curtail when predicted PRC < tight threshold
    cost_B = bt['hourly_net_cost'].where(~curtail_B, 0).sum()
    results.append({
        'strategy': 'B: PRC-only',
        'price_threshold': '-',
        'total_cost': cost_B,
        'hours_curtailed': curtail_B.sum(),
        'savings_vs_A': total_A - cost_B,
    })
    
    for thresh in price_thresholds:
        # Strategy C: Price-only
        curtail_C = (bt['pred_price'] > thresh)
        cost_C = bt['hourly_net_cost'].where(~curtail_C, 0).sum()
        results.append({
            'strategy': f'C: Price-only',
            'price_threshold': f'${thresh}',
            'total_cost': cost_C,
            'hours_curtailed': curtail_C.sum(),
            'savings_vs_A': total_A - cost_C,
        })
        
        # Strategy D: Combined (curtail on EITHER signal)
        curtail_D = curtail_B | curtail_C
        cost_D = bt['hourly_net_cost'].where(~curtail_D, 0).sum()
        results.append({
            'strategy': f'D: Combined',
            'price_threshold': f'${thresh}',
            'total_cost': cost_D,
            'hours_curtailed': curtail_D.sum(),
            'savings_vs_A': total_A - cost_D,
        })
    
    # Strategy E: Oracle
    total_E = bt['cost_E'].sum()
    results.append({
        'strategy': 'E: Oracle',
        'price_threshold': '-',
        'total_cost': total_E,
        'hours_curtailed': (bt['RT_price'] > MINING_REV).sum(),
        'savings_vs_A': total_A - total_E,
    })
    
    results_df = pd.DataFrame(results)
    results_df['pct_of_oracle'] = (
        results_df['savings_vs_A'] / results_df.loc[results_df['strategy'] == 'E: Oracle', 'savings_vs_A'].values[0] * 100
    ).round(1)
    
    return results_df

mining_results = run_mining_backtest(bt)

# Format for display
display_df = mining_results.copy()
display_df['total_cost'] = display_df['total_cost'].apply(lambda x: f'${x:,.0f}')
display_df['savings_vs_A'] = display_df['savings_vs_A'].apply(lambda x: f'${x:,.0f}')
display_df['hours_curtailed'] = display_df['hours_curtailed'].apply(lambda x: f'{x:,}')
display_df

,strategy,price_threshold,total_cost,hours_curtailed,savings_vs_A,pct_of_oracle
0,A: Always-on,-,"$-21,123,912",0,$0,0.0
1,B: PRC-only,-,"$-21,146,225",2,"$22,313",0.1
2,C: Price-only,$40,"$-38,285,498","2,772","$17,161,585",99.7
3,D: Combined,$40,"$-38,285,498","2,772","$17,161,585",99.7
4,C: Price-only,$60,"$-35,853,574","1,116","$14,729,662",85.6
5,D: Combined,$60,"$-35,855,718","1,117","$14,731,805",85.6
6,C: Price-only,$80,"$-32,657,835",558,"$11,533,923",67.0
7,D: Combined,$80,"$-32,659,979",559,"$11,536,066",67.0
8,C: Price-only,$100,"$-30,353,078",330,"$9,229,166",53.6
9,D: Combined,$100,"$-30,355,222",331,"$9,231,309",53.6


## Key comparison: B vs D (incremental value of price model)

In [6]:
# B vs D at each price threshold
print('='*70)
print('INCREMENTAL VALUE OF PRICE MODEL (Strategy D - Strategy B)')
print('='*70)

cost_B = mining_results.loc[mining_results['strategy'] == 'B: PRC-only', 'savings_vs_A'].values[0]

for _, row in mining_results[mining_results['strategy'].str.startswith('D:')].iterrows():
    incremental = row['savings_vs_A'] - cost_B
    print(f"  Price threshold {row['price_threshold']:>5}: "
          f"Combined saves ${row['savings_vs_A']:>12,.0f} vs PRC-only ${cost_B:>12,.0f} → "
          f"incremental ${incremental:>10,.0f}")

INCREMENTAL VALUE OF PRICE MODEL (Strategy D - Strategy B)
  Price threshold   $40: Combined saves $  17,161,585 vs PRC-only $      22,313 → incremental $17,139,272
  Price threshold   $60: Combined saves $  14,731,805 vs PRC-only $      22,313 → incremental $14,709,492
  Price threshold   $80: Combined saves $  11,536,066 vs PRC-only $      22,313 → incremental $11,513,753
  Price threshold  $100: Combined saves $   9,231,309 vs PRC-only $      22,313 → incremental $ 9,208,996


## False positive analysis

Hours where the model said "curtail" but you would have made money staying on.
Lost revenue = (mining_revenue - actual_price) × capacity for curtailed hours where RT < mining_rev.

In [7]:
print('='*70)
print('FALSE POSITIVE COST (unnecessary curtailment — lost mining revenue)')
print('='*70)

for thresh in [40, 60, 80, 100]:
    # Combined strategy
    curtail_D = (bt['pred_prc'] < PRC_TIGHT) | (bt['pred_price'] > thresh)
    
    # Hours curtailed where we would have made money (RT_price < mining_rev)
    false_curtail = curtail_D & (bt['RT_price'] <= MINING_REV)
    lost_revenue = ((MINING_REV - bt.loc[false_curtail, 'RT_price']) * CAPACITY_MW).sum()
    
    # Hours correctly curtailed (RT_price > mining_rev)
    true_curtail = curtail_D & (bt['RT_price'] > MINING_REV)
    avoided_cost = ((bt.loc[true_curtail, 'RT_price'] - MINING_REV) * CAPACITY_MW).sum()
    
    print(f'\n  Price threshold: ${thresh}')
    print(f'    Hours curtailed: {curtail_D.sum():,}')
    print(f'    True positives (avoided losses): {true_curtail.sum():,} hours, ${avoided_cost:,.0f} saved')
    print(f'    False positives (lost revenue):  {false_curtail.sum():,} hours, ${lost_revenue:,.0f} lost')
    print(f'    Net benefit: ${avoided_cost - lost_revenue:,.0f}')

FALSE POSITIVE COST (unnecessary curtailment — lost mining revenue)

  Price threshold: $40
    Hours curtailed: 2,772
    True positives (avoided losses): 2,654 hours, $17,193,340 saved
    False positives (lost revenue):  118 hours, $31,754 lost
    Net benefit: $17,161,585

  Price threshold: $60
    Hours curtailed: 1,117
    True positives (avoided losses): 1,115 hours, $14,737,171 saved
    False positives (lost revenue):  2 hours, $5,365 lost
    Net benefit: $14,731,805

  Price threshold: $80
    Hours curtailed: 559
    True positives (avoided losses): 558 hours, $11,538,503 saved
    False positives (lost revenue):  1 hours, $2,437 lost
    Net benefit: $11,536,066

  Price threshold: $100
    Hours curtailed: 331
    True positives (avoided losses): 330 hours, $9,233,746 saved
    False positives (lost revenue):  1 hours, $2,437 lost
    Net benefit: $9,231,309


## Backtest: Datacenter use case

- 200 MW total, 65% critical (always-on), 35% flexible (curtailable)
- Curtailment penalty: $50/MWh (SLA costs)
- On scarcity: shed all flexible load
- On tight/spike: shed half flexible load

In [8]:
CRITICAL_PCT = 0.65
CURTAIL_PENALTY = 50  # $/MWh
FLEX_MW = CAPACITY_MW * (1 - CRITICAL_PCT)  # 70 MW

def datacenter_cost(row, curtail_level):
    """Calculate hourly cost for datacenter at a given curtailment level.
    curtail_level: 0 = full power, 0.5 = half flexible shed, 1.0 = all flexible shed
    """
    load_mw = CAPACITY_MW - (FLEX_MW * curtail_level)
    reduced_mw = CAPACITY_MW - load_mw
    return (row['RT_price'] * load_mw) + (CURTAIL_PENALTY * reduced_mw)

def run_datacenter_backtest(bt, price_thresholds=[40, 60, 80, 100]):
    results = []
    
    # A: Always-on
    cost_A = (bt['RT_price'] * CAPACITY_MW).sum()
    results.append({'strategy': 'A: Always-on', 'price_threshold': '-',
                    'total_cost': cost_A, 'hours_reduced': 0, 'savings_vs_A': 0})
    
    # B: PRC-only
    scarcity_B = bt['pred_prc'] < PRC_SCARCITY
    tight_B = (bt['pred_prc'] >= PRC_SCARCITY) & (bt['pred_prc'] < PRC_TIGHT)
    
    cost_B_hourly = bt['RT_price'] * CAPACITY_MW  # default full power
    # Scarcity: shed all flexible
    load_scarcity = CAPACITY_MW * CRITICAL_PCT
    cost_B_hourly[scarcity_B] = (bt.loc[scarcity_B, 'RT_price'] * load_scarcity) + (CURTAIL_PENALTY * FLEX_MW)
    # Tight: shed half flexible
    load_tight = CAPACITY_MW - (FLEX_MW * 0.5)
    cost_B_hourly[tight_B] = (bt.loc[tight_B, 'RT_price'] * load_tight) + (CURTAIL_PENALTY * FLEX_MW * 0.5)
    
    results.append({'strategy': 'B: PRC-only', 'price_threshold': '-',
                    'total_cost': cost_B_hourly.sum(),
                    'hours_reduced': (scarcity_B | tight_B).sum(),
                    'savings_vs_A': cost_A - cost_B_hourly.sum()})
    
    for thresh in price_thresholds:
        spike_C = bt['pred_price'] > thresh
        
        # C: Price-only (shed half flexible on spike)
        cost_C_hourly = bt['RT_price'] * CAPACITY_MW
        cost_C_hourly[spike_C] = (bt.loc[spike_C, 'RT_price'] * load_tight) + (CURTAIL_PENALTY * FLEX_MW * 0.5)
        results.append({'strategy': 'C: Price-only', 'price_threshold': f'${thresh}',
                        'total_cost': cost_C_hourly.sum(),
                        'hours_reduced': spike_C.sum(),
                        'savings_vs_A': cost_A - cost_C_hourly.sum()})
        
        # D: Combined (PRC scarcity/tight OR price spike)
        cost_D_hourly = bt['RT_price'] * CAPACITY_MW
        # Scarcity from PRC: shed all flexible
        cost_D_hourly[scarcity_B] = (bt.loc[scarcity_B, 'RT_price'] * load_scarcity) + (CURTAIL_PENALTY * FLEX_MW)
        # Tight or spike: shed half flexible (don't override scarcity)
        tight_or_spike = (tight_B | spike_C) & ~scarcity_B
        cost_D_hourly[tight_or_spike] = (bt.loc[tight_or_spike, 'RT_price'] * load_tight) + (CURTAIL_PENALTY * FLEX_MW * 0.5)
        results.append({'strategy': 'D: Combined', 'price_threshold': f'${thresh}',
                        'total_cost': cost_D_hourly.sum(),
                        'hours_reduced': (scarcity_B | tight_B | spike_C).sum(),
                        'savings_vs_A': cost_A - cost_D_hourly.sum()})
    
    # E: Oracle — curtail whenever RT_price > penalty threshold
    # Shed half flexible when RT > penalty/flex equivalent
    oracle_thresh = CURTAIL_PENALTY / (1 - CRITICAL_PCT)  # ~$143/MWh — price at which shedding saves money
    oracle_curtail = bt['RT_price'] > oracle_thresh
    cost_E_hourly = bt['RT_price'] * CAPACITY_MW
    cost_E_hourly[oracle_curtail] = (bt.loc[oracle_curtail, 'RT_price'] * load_scarcity) + (CURTAIL_PENALTY * FLEX_MW)
    results.append({'strategy': 'E: Oracle', 'price_threshold': f'>${oracle_thresh:.0f}',
                    'total_cost': cost_E_hourly.sum(),
                    'hours_reduced': oracle_curtail.sum(),
                    'savings_vs_A': cost_A - cost_E_hourly.sum()})
    
    results_df = pd.DataFrame(results)
    oracle_savings = results_df.loc[results_df['strategy'] == 'E: Oracle', 'savings_vs_A'].values[0]
    results_df['pct_of_oracle'] = (results_df['savings_vs_A'] / oracle_savings * 100).round(1) if oracle_savings > 0 else 0
    return results_df

dc_results = run_datacenter_backtest(bt)

display_dc = dc_results.copy()
display_dc['total_cost'] = display_dc['total_cost'].apply(lambda x: f'${x:,.0f}')
display_dc['savings_vs_A'] = display_dc['savings_vs_A'].apply(lambda x: f'${x:,.0f}')
display_dc['hours_reduced'] = display_dc['hours_reduced'].apply(lambda x: f'{x:,}')
display_dc

,strategy,price_threshold,total_cost,hours_reduced,savings_vs_A,pct_of_oracle
0,A: Always-on,-,"$76,028,088",0,$0,0.0
1,B: PRC-only,-,"$76,024,883",2,"$3,205",0.2
2,C: Price-only,$40,"$73,995,010","2,772","$2,033,077",96.4
3,D: Combined,$40,"$73,995,010","2,772","$2,033,077",96.4
4,C: Price-only,$60,"$73,840,997","1,116","$2,187,091",103.7
5,D: Combined,$60,"$73,840,972","1,117","$2,187,116",103.7
6,C: Price-only,$80,"$74,204,951",558,"$1,823,136",86.4
7,D: Combined,$80,"$74,204,926",559,"$1,823,162",86.4
8,C: Price-only,$100,"$74,528,484",330,"$1,499,604",71.1
9,D: Combined,$100,"$74,528,459",331,"$1,499,629",71.1


## Summary: Incremental value of price model by use case

In [9]:
print('='*70)
print('SUMMARY: INCREMENTAL VALUE OF ADDING PRICE MODEL')
print('='*70)

# Mining
print('\n--- Mining (200 MW, binary curtailment) ---')
mining_B = mining_results.loc[mining_results['strategy'] == 'B: PRC-only', 'savings_vs_A'].values[0]
for _, row in mining_results[mining_results['strategy'].str.startswith('D:')].iterrows():
    inc = row['savings_vs_A'] - mining_B
    print(f"  @{row['price_threshold']:>5}: PRC-only ${mining_B:>12,.0f}  |  Combined ${row['savings_vs_A']:>12,.0f}  |  +${inc:>10,.0f}  ({row['pct_of_oracle']:.0f}% of oracle)")

# Datacenter
print('\n--- Datacenter (200 MW, 65% critical / 35% flexible) ---')
dc_B = dc_results.loc[dc_results['strategy'] == 'B: PRC-only', 'savings_vs_A'].values[0]
for _, row in dc_results[dc_results['strategy'].str.startswith('D:')].iterrows():
    inc = row['savings_vs_A'] - dc_B
    print(f"  @{row['price_threshold']:>5}: PRC-only ${dc_B:>12,.0f}  |  Combined ${row['savings_vs_A']:>12,.0f}  |  +${inc:>10,.0f}  ({row['pct_of_oracle']:.0f}% of oracle)")

print(f'\nTest period: {bt["timestamp"].min().date()} → {bt["timestamp"].max().date()} ({len(bt):,} hours)')

SUMMARY: INCREMENTAL VALUE OF ADDING PRICE MODEL

--- Mining (200 MW, binary curtailment) ---
  @  $40: PRC-only $      22,313  |  Combined $  17,161,585  |  +$17,139,272  (100% of oracle)
  @  $60: PRC-only $      22,313  |  Combined $  14,731,805  |  +$14,709,492  (86% of oracle)
  @  $80: PRC-only $      22,313  |  Combined $  11,536,066  |  +$11,513,753  (67% of oracle)
  @ $100: PRC-only $      22,313  |  Combined $   9,231,309  |  +$ 9,208,996  (54% of oracle)

--- Datacenter (200 MW, 65% critical / 35% flexible) ---
  @  $40: PRC-only $       3,205  |  Combined $   2,033,077  |  +$ 2,029,873  (96% of oracle)
  @  $60: PRC-only $       3,205  |  Combined $   2,187,116  |  +$ 2,183,911  (104% of oracle)
  @  $80: PRC-only $       3,205  |  Combined $   1,823,162  |  +$ 1,819,957  (86% of oracle)
  @ $100: PRC-only $       3,205  |  Combined $   1,499,629  |  +$ 1,496,424  (71% of oracle)

Test period: 2024-07-01 → 2025-12-04 (12,144 hours)
